# Parte 3 — Ecuaciones elípticas
## 3.5 El problema de Dirichlet
### 3.5.01 Formulación clásica, unicidad, soluciones explícitas y puntos regulares

Este notebook cubre un **hueco del temario oficial** que no aparece desarrollado
como sección independiente en las páginas manuscritas disponibles.

## Base tomada de las notas

Las páginas 8 y 9 de las notas proporcionan los ingredientes inmediatos:

- función de Green;
- núcleo de Poisson;
- representación de soluciones;
- principio del máximo y unicidad desarrollados en notebooks anteriores.

No se atribuye a las notas ningún enunciado adicional que no aparezca allí.
Todo el desarrollo específico del problema de Dirichlet se marca como
**Complemento para cerrar el temario oficial**.

## Contenido

1. formulación clásica del problema de Dirichlet;
2. unicidad mediante el principio del máximo;
3. existencia explícita en la bola;
4. existencia explícita en el semiespacio;
5. concentración del núcleo de Poisson y recuperación del dato;
6. puntos regulares y barreras;
7. condición de esfera exterior.

El método de Perron se reserva para el subcapítulo 3.7.

## Páginas de las notas utilizadas como fundamento

### Página 8

![Página 8 — inicio de función de Green](../fuentes/pagina_08_notas.png)

### Página 9

![Página 9 — Green y estimaciones](../fuentes/pagina_09_notas.png)

Estas páginas sustentan las fórmulas de Green y Poisson usadas aquí. La sección
3.5 es una ampliación explícita para completar el programa oficial.

# Simulaciones y visualizaciones

Las celdas se ejecutan directamente. No existe una bandera `VIDEO=True`.

Se incluyen:

1. animación de concentración del núcleo de Poisson en el disco;
2. recuperación de un dato continuo de frontera;
3. reconstrucción en el semiespacio;
4. visualización de una barrera producida por una esfera exterior;
5. comparación entre el dato de frontera y valores interiores cercanos a la frontera.

CuPy/CUDA se usa automáticamente en mallas y barridos densos cuando está disponible.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend numérico: CuPy/CUDA")
    else:
        raise RuntimeError("No se encontró un dispositivo CUDA.")
except Exception as exc:
    xp = np
    print("Backend numérico: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def to_cpu(array):
    if GPU_AVAILABLE:
        return cp.asnumpy(array)
    return np.asarray(array)


def save_and_display_animation(
    animation,
    stem,
    fps=60,
    dpi=150,
    bitrate=10000,
):
    """Guarda y muestra automáticamente una animación."""
    if shutil.which("ffmpeg"):
        output = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(output, writer=writer, dpi=dpi)
        display(Video(str(output), embed=True))
    else:
        output = ANIM_DIR / f"{stem}.gif"
        writer = PillowWriter(fps=min(fps, 35))
        animation.save(
            output,
            writer=writer,
            dpi=min(dpi, 110),
        )
        display(Image(filename=str(output)))

    print("Animación guardada en:", output.resolve())
    return output

## Simulación 3.5.A — Concentración del núcleo de Poisson

En el disco unitario, si el punto interior es

$$
y=r(\cos\varphi_0,\sin\varphi_0),
$$

el núcleo de Poisson es

$$
P_r(\theta-\varphi_0)
=
\frac{1-r^2}
{2\pi\left(1-2r\cos(\theta-\varphi_0)+r^2\right)}.
$$

Cuando $r\uparrow1$, la masa se concentra cerca de $\theta=\varphi_0$, pero
la integral total permanece igual a uno.

In [ ]:
# ============================================================
# CONCENTRACIÓN DEL NÚCLEO DE POISSON
# ============================================================

n_theta = 5000 if GPU_AVAILABLE else 2400
theta = xp.linspace(
    -math.pi,
    math.pi,
    n_theta,
    endpoint=False,
)
theta_cpu = to_cpu(theta)

phi0 = 0.65
radii = np.linspace(0.02, 0.995, 240)

kernels = []

for radius in radii:
    kernel = (
        (1.0 - radius**2)
        / (
            2.0
            * math.pi
            * (
                1.0
                - 2.0 * radius * xp.cos(theta - phi0)
                + radius**2
            )
        )
    )
    kernels.append(to_cpu(kernel).astype(np.float32))

kernels = np.asarray(kernels)

dtheta = float(theta_cpu[1] - theta_cpu[0])
masses = np.sum(kernels, axis=1) * dtheta

print("Masa mínima:", masses.min())
print("Masa máxima:", masses.max())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

line, = ax.plot(theta_cpu, kernels[0])
ax.set_xlim(-math.pi, math.pi)
ax.set_ylim(0.0, float(np.max(kernels)) * 1.03)
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$P_r(\theta-\varphi_0)$")
ax.set_title("Concentración del núcleo de Poisson")

status = ax.text(
    0.02,
    0.96,
    "",
    transform=ax.transAxes,
    va="top",
)


def update_poisson_kernel(frame):
    line.set_data(theta_cpu, kernels[frame])
    status.set_text(
        rf"$r={radii[frame]:.3f}$"
        + "\n"
        + rf"$\int P_r={masses[frame]:.6f}$"
    )
    return line, status


animation = FuncAnimation(
    fig,
    update_poisson_kernel,
    frames=len(radii),
    interval=1000.0 / 60.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation,
    "03.5.A_concentracion_nucleo_poisson",
    fps=60,
    dpi=160,
    bitrate=12000,
)

plt.close(fig)

## Simulación 3.5.B — Recuperación de un dato continuo

Tomamos

$$
g(\theta)
=
0.8\cos(2\theta)
+
0.35\sin(5\theta)
+
0.2\cos(11\theta).
$$

La integral de Poisson produce

$$
u(r,\varphi)
=
0.8r^2\cos(2\varphi)
+
0.35r^5\sin(5\varphi)
+
0.2r^{11}\cos(11\varphi).
$$

Se muestra cómo $u(r,\cdot)$ converge uniformemente a $g$ cuando $r\uparrow1$.

In [ ]:
g_boundary = (
    0.8 * np.cos(2.0 * theta_cpu)
    + 0.35 * np.sin(5.0 * theta_cpu)
    + 0.2 * np.cos(11.0 * theta_cpu)
)

radii_recovery = np.linspace(0.02, 0.999, 220)
profiles = []
uniform_errors = []

for radius in radii_recovery:
    profile = (
        0.8 * radius**2 * np.cos(2.0 * theta_cpu)
        + 0.35 * radius**5 * np.sin(5.0 * theta_cpu)
        + 0.2 * radius**11 * np.cos(11.0 * theta_cpu)
    )
    profiles.append(profile.astype(np.float32))
    uniform_errors.append(
        float(np.max(np.abs(profile - g_boundary)))
    )

profiles = np.asarray(profiles)
uniform_errors = np.asarray(uniform_errors)

fig, ax = plt.subplots(figsize=(10, 6))

boundary_line, = ax.plot(
    theta_cpu,
    g_boundary,
    linestyle="--",
    label=r"$g(\theta)$",
)
interior_line, = ax.plot(
    theta_cpu,
    profiles[0],
    label=r"$u(r,\theta)$",
)

ax.set_xlim(-math.pi, math.pi)
ax.set_ylim(
    min(g_boundary.min(), profiles.min()) - 0.08,
    max(g_boundary.max(), profiles.max()) + 0.08,
)
ax.set_xlabel(r"$\theta$")
ax.set_ylabel("valor")
ax.set_title("Recuperación del dato de Dirichlet")
ax.legend()

status = ax.text(
    0.02,
    0.96,
    "",
    transform=ax.transAxes,
    va="top",
)


def update_recovery(frame):
    interior_line.set_data(theta_cpu, profiles[frame])
    status.set_text(
        rf"$r={radii_recovery[frame]:.3f}$"
        + "\n"
        + rf"$\|u(r,\cdot)-g\|_\infty={uniform_errors[frame]:.3e}$"
    )
    return boundary_line, interior_line, status


animation_recovery = FuncAnimation(
    fig,
    update_recovery,
    frames=len(radii_recovery),
    interval=1000.0 / 60.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation_recovery,
    "03.5.B_recuperacion_dato_continuo",
    fps=60,
    dpi=160,
    bitrate=12000,
)

plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(
    radii_recovery,
    uniform_errors,
)
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\|u(r,\cdot)-g\|_\infty$")
ax.set_title("Convergencia uniforme hacia el dato de frontera")
ax.grid(True, alpha=0.3)
fig.tight_layout()

path = FIG_DIR / "03.5.B_error_recuperacion.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.5.C — Fórmula de Poisson en el semiespacio

Para

$$
\mathbb H
=
\{(x',x_n)\in\mathbb R^n:x_n>0\},
$$

el núcleo es

$$
P(x_n,x'-\xi)
=
\frac{2}{\omega_n}
\frac{x_n}
{\left(|x'-\xi|^2+x_n^2\right)^{n/2}}.
$$

En dimensión dos, con dato

$$
g(\xi)=e^{-\xi^2}\cos(3\xi),
$$

se calcula la integral numérica para alturas decrecientes.

In [ ]:
# ============================================================
# SEMIPLANO EN DIMENSIÓN DOS
# ============================================================

xi = np.linspace(-12.0, 12.0, 16000)
dxi = xi[1] - xi[0]
g_half = np.exp(-xi**2) * np.cos(3.0 * xi)

x_eval = np.linspace(-4.0, 4.0, 900)
heights = (1.0, 0.55, 0.25, 0.10, 0.045)

profiles_half = []

for height in heights:
    values = np.empty_like(x_eval)

    for index, x0 in enumerate(x_eval):
        kernel = (
            1.0
            / math.pi
            * height
            / ((x0 - xi) ** 2 + height**2)
        )
        values[index] = np.trapz(
            kernel * g_half,
            xi,
        )

    profiles_half.append(values)

profiles_half = np.asarray(profiles_half)
g_eval = np.exp(-x_eval**2) * np.cos(3.0 * x_eval)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    x_eval,
    g_eval,
    linewidth=2.0,
    label=r"$g(x')$",
)

for height, values in zip(heights, profiles_half):
    ax.plot(
        x_eval,
        values,
        label=rf"$x_n={height}$",
    )

ax.set_xlabel(r"$x'$")
ax.set_ylabel("valor")
ax.set_title("Fórmula de Poisson en el semiplano")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.5.C_poisson_semiplano.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.5.D — Barrera por esfera exterior

Supongamos que una bola exterior

$$
\overline{B_R(z)}
$$

toca la frontera de $\Omega$ sólo en $\xi$.

En dimensión dos,

$$
w(x)=\log\frac{|x-z|}{R}
$$

es armónica fuera de $z$, no negativa en la región exterior a la bola y satisface

$$
w(x)\to0
\qquad\text{cuando }x\to\xi.
$$

Esta función sirve como barrera en el punto de contacto.

In [ ]:
# ============================================================
# GEOMETRÍA DE UNA BARRERA EXTERIOR
# ============================================================

grid_n = 700 if GPU_AVAILABLE else 380
axis = xp.linspace(-1.9, 2.1, grid_n)
BX, BY = xp.meshgrid(axis, axis, indexing="xy")

# Dominio ilustrativo: disco unitario.
domain_mask = BX**2 + BY**2 < 1.0

# Bola exterior tangente en xi=(1,0).
R_ext = 0.65
center_ext = (1.0 + R_ext, 0.0)
distance_ext = xp.sqrt(
    (BX - center_ext[0]) ** 2
    + (BY - center_ext[1]) ** 2
)

barrier = xp.log(
    xp.maximum(distance_ext, 1e-10) / R_ext
)
barrier_domain = xp.where(
    domain_mask,
    barrier,
    xp.nan,
)

barrier_cpu = to_cpu(barrier_domain)

fig, ax = plt.subplots(figsize=(8.5, 7))
image = ax.imshow(
    barrier_cpu,
    origin="lower",
    extent=[
        axis.get()[0] if GPU_AVAILABLE else axis[0],
        axis.get()[-1] if GPU_AVAILABLE else axis[-1],
        axis.get()[0] if GPU_AVAILABLE else axis[0],
        axis.get()[-1] if GPU_AVAILABLE else axis[-1],
    ],
    interpolation="bilinear",
)
fig.colorbar(image, ax=ax, label=r"$w(x)$")

domain_circle = plt.Circle(
    (0.0, 0.0),
    1.0,
    fill=False,
    linewidth=1.8,
)
exterior_circle = plt.Circle(
    center_ext,
    R_ext,
    fill=False,
    linestyle="--",
    linewidth=1.8,
)
ax.add_patch(domain_circle)
ax.add_patch(exterior_circle)
ax.plot([1.0], [0.0], marker="o", label=r"$\xi$")
ax.set_aspect("equal")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title("Barrera inducida por una esfera exterior")
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.5.D_barrera_esfera_exterior.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print("Mínimo de la barrera en el dominio:", np.nanmin(barrier_cpu))
print(path.resolve())

# 3.5.1 Formulación clásica

## **Definición 3.5.1 (Problema de Dirichlet para la ecuación de Laplace).**

Sea $\Omega\subset\mathbb R^n$ un dominio y sea

$$
g:\partial\Omega\longrightarrow\mathbb R.
$$

El **problema de Dirichlet** consiste en encontrar una función $u$ tal que

$$
\begin{cases}
\Delta u=0,
& \text{en }\Omega,\\
u=g,
& \text{sobre }\partial\Omega.
\end{cases}
$$

## **Definición 3.5.2 (Solución clásica del problema de Dirichlet).**

Si $\Omega$ es acotado y $g\in C(\partial\Omega)$, una solución clásica es una
función

$$
u\in C^2(\Omega)\cap C(\overline\Omega)
$$

que satisface la ecuación y el dato de frontera punto por punto.

## **Definición 3.5.3 (Solvencia clásica).**

Se dice que el problema de Dirichlet es **clásicamente soluble** en $\Omega$ si
para todo $g\in C(\partial\Omega)$ existe una solución clásica.

### **Complemento para cerrar el temario oficial.**

La continuidad de $g$ no garantiza por sí sola la existencia en un dominio
arbitrario. La geometría de la frontera interviene. La unicidad, en cambio, se
obtiene del principio del máximo bajo hipótesis muy generales.

## **Teorema 3.5.4 (Unicidad del problema de Dirichlet).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado. Si

$$
u_1,u_2\in C^2(\Omega)\cap C(\overline\Omega)
$$

satisfacen

$$
\Delta u_1=\Delta u_2=0
$$

en $\Omega$ y

$$
u_1=u_2
$$

sobre $\partial\Omega$, entonces

$$
u_1\equiv u_2
$$

en $\overline\Omega$.

### Demostración

La diferencia

$$
w=u_1-u_2
$$

es armónica y satisface $w=0$ en la frontera. El principio del máximo aplicado
a $w$ y a $-w$ da

$$
w\leq0
\quad\text{y}\quad
-w\leq0.
$$

Por tanto, $w=0$.

$\square$

## **Corolario 3.5.5 (Estabilidad respecto del dato).**

Si $u_i$ es la solución correspondiente a $g_i$, entonces

$$
\|u_1-u_2\|_{L^\infty(\Omega)}
\leq
\|g_1-g_2\|_{L^\infty(\partial\Omega)}.
$$

### Ejercicios — Sección 3.5.1

1. Demuestre el Corolario 3.5.5 mediante comparación.

2. Explique por qué la acotación del dominio aparece en la prueba elemental de
   unicidad.

3. **Tipo Examen General.** Sea $\Omega=\mathbb R^n\setminus\overline{B_R}$.
   Determine condiciones en el infinito que garanticen unicidad para un problema
   exterior de Dirichlet en dimensiones $n=2$ y $n\geq3$.

# 3.5.2 Existencia explícita en la bola

## **Teorema 3.5.6 (Fórmula de Poisson para la bola).**

Sea

$$
B_R(x_0)
=
\{x\in\mathbb R^n:|x-x_0|<R\}
$$

y sea $g\in C(\partial B_R(x_0))$.

Definimos

$$
u(x)
=
\int_{\partial B_R(x_0)}
P_R(x,\xi)g(\xi)\,dS_\xi,
$$

donde

$$
P_R(x,\xi)
=
\frac{R^2-|x-x_0|^2}
{\omega_nR|x-\xi|^n}.
$$

Entonces

$$
u\in C^\infty(B_R(x_0))\cap C(\overline{B_R(x_0)})
$$

y

$$
\begin{cases}
\Delta u=0,
& \text{en }B_R(x_0),\\
u=g,
& \text{sobre }\partial B_R(x_0).
\end{cases}
$$

### Demostración añadida

Para cada $\xi\in\partial B_R(x_0)$, la función $P_R(\cdot,\xi)$ es armónica
en la bola. La diferenciación bajo el signo integral prueba $\Delta u=0$.

Además,

$$
P_R(x,\xi)\geq0
$$

y

$$
\int_{\partial B_R(x_0)}P_R(x,\xi)\,dS_\xi=1.
$$

Fijemos $\xi_0\in\partial B_R(x_0)$. Escribimos

$$
u(x)-g(\xi_0)
=
\int_{\partial B_R}
P_R(x,\xi)
\left(g(\xi)-g(\xi_0)\right)dS_\xi.
$$

La continuidad uniforme de $g$ controla la integral cerca de $\xi_0$, mientras
que la masa del núcleo lejos de $\xi_0$ tiende a cero cuando $x\to\xi_0$.
Por tanto,

$$
u(x)\to g(\xi_0).
$$

$\square$

## **Corolario 3.5.7 (Propiedades de la extensión de Poisson).**

Bajo las hipótesis del Teorema 3.5.6:

1. si $g\geq0$, entonces $u\geq0$;
2. si $g_1\leq g_2$, entonces $u_1\leq u_2$;
3. se cumple
   $$
   \min_{\partial B_R}g
   \leq
   u(x)
   \leq
   \max_{\partial B_R}g;
   $$
4. si $g$ no es constante, las desigualdades son estrictas en el interior.

### Ejercicios — Sección 3.5.2

1. Verifique que $P_R$ tiene integral uno.

2. Deduzca la fórmula de la bola de radio $R$ a partir de la bola unitaria por
   traslación y escalamiento.

3. Calcule la extensión de Poisson de un armónico esférico de grado $k$.

4. **Tipo Examen General.** Sea $g\in C(\partial B_1)$ y sea $u$ su extensión
   de Poisson. Pruebe directamente, sin usar el principio del máximo, que
   $\|u\|_\infty\leq\|g\|_\infty$.

# 3.5.3 Existencia explícita en el semiespacio

## **Teorema 3.5.8 (Fórmula de Poisson para el semiespacio).**

Sea

$$
\mathbb H
=
\{(x',x_n)\in\mathbb R^{n-1}\times\mathbb R:x_n>0\}.
$$

Supongamos que $g\in C_b(\mathbb R^{n-1})$. Definimos

$$
u(x',x_n)
=
\int_{\mathbb R^{n-1}}
P(x_n,x'-\xi)g(\xi)\,d\xi,
$$

donde

$$
P(x_n,z)
=
\frac{2}{\omega_n}
\frac{x_n}
{\left(|z|^2+x_n^2\right)^{n/2}}.
$$

Entonces $u$ es armónica en $\mathbb H$, está acotada y satisface

$$
u(x',x_n)\to g(x'_0)
$$

cuando $(x',x_n)\to(x'_0,0)$.

### Complemento

El núcleo es no negativo, tiene integral uno y forma una identidad aproximada
cuando $x_n\downarrow0$.

### Ejercicios — Sección 3.5.3

1. Demuestre que el núcleo del semiespacio tiene integral uno.

2. Verifique directamente su invariancia bajo traslaciones tangenciales y
   escalamiento.

3. Estudie qué hipótesis de crecimiento pueden sustituir a $g\in C_b$.

4. **Tipo Examen General.** Use el método de imágenes y la derivada normal de
   Green para derivar la fórmula de Poisson del semiespacio.

# 3.5.4 Puntos regulares y barreras

## **Definición 3.5.9 (Punto regular para el problema de Dirichlet).**

Sea $\Omega$ un dominio acotado y sea $\xi\in\partial\Omega$.

El punto $\xi$ se llama **regular** si, para todo $g\in C(\partial\Omega)$, la
solución generalizada de Dirichlet satisface

$$
\lim_{\substack{x\to\xi\\x\in\Omega}}u(x)=g(\xi).
$$

### **Aclaración.**

La construcción de la “solución generalizada” mediante Perron se desarrollará
en el subcapítulo 3.7. Aquí se estudia el criterio geométrico mediante barreras.

## **Definición 3.5.10 (Barrera en un punto de frontera).**

Una función $w$ es una **barrera en $\xi\in\partial\Omega$** si existe un entorno
$V$ de $\xi$ tal que:

1. $w$ es superarmónica en $\Omega\cap V$;
2. $w>0$ en $\Omega\cap V$;
3. $w(x)\to0$ cuando $x\to\xi$ desde $\Omega$;
4. para todo entorno $W$ de $\xi$ con $\overline W\subset V$,
   $w$ está separada de cero en
   $$
   \Omega\cap\partial W.
   $$

## **Teorema 3.5.11 (Criterio de barrera).**

Si existe una barrera en $\xi$, entonces $\xi$ es un punto regular.

### Comentario

La prueba completa utiliza comparación con las funciones de Perron. Se dará
formalmente en el notebook del método de Perron.

## **Definición 3.5.12 (Condición de esfera exterior).**

Se dice que $\Omega$ satisface la **condición de esfera exterior en $\xi$** si
existen $z\notin\Omega$ y $R>0$ tales que

$$
\overline{B_R(z)}\cap\overline\Omega=\{\xi\}.
$$

## **Teorema 3.5.13 (Regularidad bajo la condición de esfera exterior).**

Si $\Omega$ satisface la condición de esfera exterior en $\xi$, entonces $\xi$
es regular.

### Demostración añadida

Si $n\geq3$, definimos

$$
w(x)
=
R^{2-n}-|x-z|^{2-n}.
$$

Como $|x-z|\geq R$ para $x\in\Omega$,

$$
w(x)\geq0.
$$

Además, $w$ es armónica fuera de $z$, se anula únicamente en $\xi$ dentro del
entorno de contacto y es positiva lejos de $\xi$.

Si $n=2$, usamos

$$
w(x)
=
\log\frac{|x-z|}{R}.
$$

La misma argumentación muestra que $w$ es una barrera. El Teorema 3.5.11 implica
la regularidad.

$\square$

## **Corolario 3.5.14.**

Todo punto de la frontera de un dominio de clase $C^2$ satisface localmente una
condición de esfera exterior y, por tanto, es regular.

### Ejercicios — Sección 3.5.4

1. Verifique todas las propiedades de la barrera construida en el
   Teorema 3.5.13.

2. Construya una barrera explícita para un semiespacio y para una bola.

3. Explique por qué la continuidad de la frontera no basta, por sí sola, para
   garantizar regularidad en todas las dimensiones.

4. **Tipo Examen General.** Sea $\Omega$ un dominio acotado que satisface la
   condición de esfera exterior en cada punto. Explique cómo las barreras y el
   principio de comparación permiten recuperar continuamente un dato
   $g\in C(\partial\Omega)$.

# Control de cobertura y estado del capítulo

## Contenido cubierto

- formulación clásica del problema de Dirichlet;
- unicidad y estabilidad;
- existencia explícita en bolas;
- existencia explícita en el semiespacio;
- núcleos de Poisson como identidades aproximadas;
- puntos regulares;
- barreras;
- condición de esfera exterior.

## Relación con las notas

Las notas disponibles no desarrollan una sección independiente de Dirichlet.
Se usaron únicamente sus fórmulas de Green y Poisson como fundamento. Todo lo
demás quedó identificado como complemento necesario para cubrir el temario
oficial.

## Pendiente inmediato

El siguiente notebook de la cola es

$$
\texttt{03.6.01\_Dirichlet\_en\_cuadrado\_series\_de\_Fourier.ipynb}.
$$

Allí se resolverá el problema de Dirichlet en un cuadrado mediante separación
de variables y series de Fourier.